In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
#while ingesting customer data from an external source, you notice duplicate enteries.How would you remove duplicate and retain only the latest entry based on a timestamp col.
data = [("101","2023-12-01",100),
        ("101","2023-12-02",150),
        ("102","2023-12-01",200),
        ("102","2023-12-02",250)]
columns = ["product_id","date","sales"]
df = spark.createDataFrame(data,columns)
display(df)

#in pyspark we write date in string fromat agar date format m likha bhi to bhi schma m datatype string hi aega
df = df.withColumn("date",col("date").cast(DateType()))

#Droping duplicate using dropduplicate function
df = df.orderBy("product_id","date",ascending=[1,0]).dropDuplicates(["product_id"]).display()
# "I wouldn't prefer dropDuplicates() for this use case because it doesn't guarantee which duplicate record will be retained, so the latest record could potentially be dropped. At production level, I would use a Window function with row_number(), partition by the unique key and order by the timestamp descending. Then I would filter row_number = 1 to retain only the latest record."

In [0]:
# Que. You are working with a real time data pipeline, and you notice missing values in your streaming data column - Category. How would you handle null or missing values in such a scenario?
df = df.fillNa({‘Category’:’N/A})

# Note:Hum aise dictionary mai different columns ke liea kr skte hain (chaye N/a do ya kuch bhi)
df = df.fillna({ "category": "N/A", "status": "Not present" }) 


In [0]:
# Que. You need to calculate the total number of actions performed by users in a system. How would you calculate the top 5 most active users based on this info?
from pyspark.sql.functions import *
from pyspark.sql.types import *
data = [("user1",5),("user2",8),("user3",3),("user4",10),("user2",3)]
col = ["user_id","actions"]
df = spark.createDataFrame(data,col)
display(df)

df = df.groupBy('user_id').agg(sum('actions').alias('total_actions')).orderBy('total_actions',ascending=False).limit(5)
display(df)

In [0]:
# Que. While processing sales transaction data, you need to identify the most recent transaction for each customer.How would you approach this task?
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window

data = [("cust1","2023-12-01",100),("cust2","2023-12-02",150),("cust1","2023-12-03",200),("cust2","2023-12-04",250)]
column = ["customer_id","transaction_date","sales"]
df = spark.createDataFrame(data,column)
df = df.withColumn("transaction_date",col("transaction_date").cast(DateType()))
window_def = Window.partitionBy("customer_id").orderBy(col("transaction_date").desc())
df_res = df.withColumn("rnk",row_number().over(window_def)).filter(col("rnk")==1).drop("rnk")
display(df_res)

In [0]:
# Que. Find customers who haven’t made any purchase in last 30 days.
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window
data = [("cust1","2026-08-20"),("cust2","2026-03-02"),("cust3","2023-11-25")]
columns = ["customer_id","last_purchase_date"]
df = spark.createDataFrame(data,columns)
display(df)

df = df.withColumn("last_purchase_date",col('last_purchase_date').cast(DateType()))
df = df.withColumn("gap", datediff(current_date(),'last_purchase_date')).filter(col("gap")>30).drop("gap")
display(df)

In [0]:
# While analysing the customer reviews, you need to identify the most frequently used words in the feedback. 
from pyspark.sql.functions import *
from pyspark.sql.types import *
data = [("customer1","The product is great"),("customer2","Great product, fast delivery"),("customer3","Not bad,could be better")]
columns = ["customer_id","feedback"]
df = spark.createDataFrame(data, columns)
display(df)

# for this we will spilt each word and count its frequency for those same words but one is written in captital letter and other at small letter to resolve this we will use lower(convert words to lower case)
df = df.withColumn('feedback',lower('feedback')).withColumn ('feedback',explode(split('feedback',' ')))
df_grp = df.groupBy('feedback').agg(count('feedback').alias('wordCount')).orderBy(col('wordCount').desc()).limit(1)
display(df_grp)

In [0]:
# Calculate cumulative sum of sales over time for each product
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window

data = [("product1","2023-12-01",100),("product2","2023-12-02",200),("product1","2023-12-03",150),("product2","2023-12-04",250)]
columns = ["product_id","date","sales"]
df = spark.createDataFrame(data,columns)
display(df)

df = df.withColumn("date",col("date").cast(DateType()))
window_def = Window.partitionBy("product_id").orderBy(col("date").asc())
res_df = df.withColumn("cumulative_sum",sum("sales").over(window_def))
display(res_df)

In [0]:
#  While preparing a data pipeline, you notice some duplicate rows in a dataset. How would you remove the duplicates without affecting the original order?
from pyspark.sql import Window
data = [("John",25),("Jane",30),("John",25),("Alice",22)]
columns = ["name","age"]
df = spark.createDataFrame(data,columns)
display(df)

res_df = df.withColumn("rnk",row_number().over(Window.partitionBy("name").orderBy('age'))).filter(col("rnk")==1).drop("rnk")
display(res_df)

In [0]:
# Calculate the average session duration per user
from pyspark.sql.functions import *
data = [("user1","2023-12-01",50),("user1","2023-12-02",60),("user2","2023-12-01",45),("user2","2023-12-03",75)]
columns = ["user_id","session_date","duration"]
df = spark.createDataFrame(data,columns)
df = df.groupBy("user_id").agg(avg("duration").alias("avg_duration"))
display(df)

In [0]:
# Find the product with the highest sales for each month.
from pyspark.sql.functions import *
from pyspark.sql.types import *
data = [("product1","2023-12-01",100),("product1","2023-10-01",60),("product2","2023-12-01",150),("product1","2023-12-02",200),("product2","2023-12-02",250)]
columns = ["product_id","date","sales"]
df = spark.createDataFrame(data,columns)
df = df.withColumn("date",col("date").cast(DateType()))
df_res = df.withColumn("months",month("date")).groupBy("months","product_id").agg(sum("sales").alias("total_sales"))
display(df_res)

In [0]:
#  Que. You have a dataset containing the names of employees and their department. You need to find the department with the most employees.
#Doing this using dense_Rank will be better option because no. of employee for 2 or more dept can be same and max at same time
from pyspark.sql.functions import *
data = [("Alice","HR"),("Ram","Finance"),("Naina","IT"),("Anu","IT")]
columns = ["name","department"]
df = spark.createDataFrame(data,columns)
df_res = df.groupBy("department").agg(count("name").alias("count")).sort("count",ascending=False).limit(1)
display(df_res)

In [0]:
# Que. While processing sales data, you need to classify each transaction as either “High” or “Low” based on its amount.
from pyspark.sql.functions import *

data = [("product1",100),("product2",300),("product3",50)]
columns = ["product_id","sales"]
df = spark.createDataFrame(data,columns)

#we will do this with case statment
df_res = df.withColumn("price_cat",when(col("sales")>50,"Hight").otherwise("Low"))
display(df_res)

In [0]:
# Que. While analysing a larger dataset, you need to create a new column that holds a timestamp of when the record was processed.
df_res = df.withColumn("processed_time",current_timestamp())
display(df_res)

In [0]:
# Que. You need to register this pyspark DF as a temporary SQL object and run a query on it. 
from pyspark.sql.functions import *
data = [("product1",100),("product2",300),("product3",50)]
columns = ["product_id","sales"]
df = spark.createDataFrame(data,columns)

#for this we can create a temp view
df.createOrReplaceTempView("tempsqldf")

#I can use this view in 2 pays
#1st using python language
spark.sql("SELECT * FROM tempsqldf")

#2nd using SQL
SELECT * FROM tempsqldf;

In [0]:
# Que. You need to query data from a Pyspark DF using SQL, but the data includes a nested structure. How would you flatten the data easier querying?
data = [("product1",{"price":100,"quantity":2}), ("product2",{"price":200, "quantity":3})]
columns = ["product_id","product_info"]
df = spark.createDataFrame(data,columns)
display(df)

df.select("product_id","product_info.price","product_info.quantity").display()

#or can create a view on top of this
df.select("product_id","product_info.price","product_info.quantity").createOrReplaceTempView("flatView")

In [0]:
%sql
SELECT * FROM flatView;

In [0]:
# Que. Your are processing a sales data.Groupby product categories and create a list of all product names in each category.
from pyspark.sql.functions import *
data = [("Electronics","Laptop"),("Electronics","Smartphone"),("Furniture","Chair"),("Furniture","Table")]
columns = ["category","product"]
df = spark.createDataFrame(data,columns)
df_res = df.groupby("category").agg(collect_list("product").alias("products"))
display(df_res)

In [0]:
# Que. You are analysing the orders. GroupBy customer IDs and list all unique product IDs each customer purchased.
from pyspark.sql.functions import *
data = [(101,"P001"),(101,"P002"),(102,"P001"),(101,"P001")]
columns = ["customer_id","product_id"]
df = spark.createDataFrame(data,columns)

df_res = df.groupby("customer_id").agg(collect_set("product_id").alias('unique_products'))
display(df_res)

In [0]:
# Que. For customer records, combine first and last names only if the email address exists.
from pyspark.sql.functions import *
data = [("John","Deo","john@gmail.com"),("Jane","Smith",None)]
columns = ["first_name","last_name","email"]
df = spark.createDataFrame(data,columns)
df_res = df.withColumn("full_name",when(col("email").isNotNull(),concat_ws(" ",col("first_name"),col("last_name"))).otherwise(None))
display(df_res)

In [0]:
# Que. You have a DF containing customerID and list of their purchase product IDs. Calculate the no. of products each customer has purchased.
data = [(1,["prod1","prod2","prod3"]),(2,["prod4"]),(3,["prod5","prod6"])]
myschema = "customer_id INT, product_ids array<STRING>"
df = spark.createDataFrame(data, myschema)

df_res = df.withColumn("num_of_products",size(col("product_ids")))
display(df_res)

In [0]:
# Que. You have employee IDs of varying lengths. Ensure all IDs are 6 characters long by padding with leading zeroes.
data =  [("1",),("123",),("4567",),]
schema = ["employee_id"]
df = spark.createDataFrame(data, schema)
df = df.withColumn("employee_id",lpad(col("employee_id"),6,"0"))
df.display()

In [0]:
# Que. You need to validate phone numbers by checking if they start with “91”.
from pyspark.sql.functions import *
data = [("911234567890",),("819105592206",),("919036799244",),]
schema = ["phone_number"]
df = spark.createDataFrame(data,schema)
df.filter(substring(col("phone_number"),1,2)=="91").display()